In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import time
import torch
import random
import numpy as np
from glob import glob
from pathlib import Path
from src.utilities.report import Report
from src.service.fragment.net import Net
from src.utilities.score_mapper import ScoreMapper
from src.utilities.load_dataset import load_dataset
from skl2onnx.helpers.onnx_helper import load_onnx_model
from src.service.stitching.generate_networks import generate_networks
from src.utilities.dataloader_generator import generate_dataloader

In [3]:
netsFiles = sorted(glob('_results/fragments/net*'))
nets = []
for index, netsFile in enumerate(netsFiles):
    fragmentFiles = sorted(glob(str(Path(netsFile)/'fragment*.onnx')))
    onnxFragments = []
    for fragmentFile in fragmentFiles:
        onnxFragment = load_onnx_model(fragmentFile)
        onnxFragments.append(onnxFragment)
    net1 = Net(onnxFragments, index)
    nets.append(net1)

In [4]:
random.seed(51)
np.random.seed(24)
torch.manual_seed(77)

K = 5
STITCH_BATCH_SIZE = 32 # todo study the effect
MAX_DEPTH = 16
THRESOULD = 0
TOTAL_THRESOULD = 0.5

RESULT_NAME = f"{int(time.time())}_result_BS_{STITCH_BATCH_SIZE}_MD_{MAX_DEPTH}_T_{THRESOULD}_TT_{TOTAL_THRESOULD}_K_{K}"

EVAL_BATCH_SIZE = 64

train_datalader = generate_dataloader(load_dataset())
test_datalader = generate_dataloader(load_dataset("test"), batch_size=EVAL_BATCH_SIZE)
data_score, _ = next(iter(train_datalader))
data_score = data_score.numpy()
print(data_score.shape)

(64, 3, 224, 224)


In [5]:
# range(1)
k = 0
if os.path.exists(f'./_results/{RESULT_NAME}.txt'):
    with open(f'./_results/{RESULT_NAME}.txt', 'r') as f:
        k = len(f.read().split('\n'))
        print(k)

In [ ]:
scoreMapper = ScoreMapper(nets, data_score)
with Report(EVAL_BATCH_SIZE, f'./_results/{RESULT_NAME}.txt', 'a') as report:
    generator = generate_networks(nets, scoreMapper, data_score, 
                          threshold=THRESOULD, totalThreshold=TOTAL_THRESOULD, 
                          maxDepth=MAX_DEPTH, sample=False, K=K)
    for i,(s,net) in enumerate(generator):
        try:
            netname = f"_results/{RESULT_NAME}/net{k:03}"
            report.evaluate(nets, net, netname, s, test_datalader)
            net.save(netname)
            k += 1
        except Exception as e:
            print('ERROR', e)
            pass